In [11]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

import mlflow

In [45]:
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")
mlflow.set_experiment("Fraud_Detection_Hybrid_Model")

<Experiment: artifact_location='/home/workstation-p/Downloads/Projects/Hybrid-Approach-for-Digital-Fraud-Detection-in-Online-Financial-Transactions/mlruns/1', creation_time=1776631581517, experiment_id='1', last_update_time=1776631581517, lifecycle_stage='active', name='Fraud_Detection_Hybrid_Model', tags={}, workspace='default'>

In [13]:
def reduce_mem_usage(df):
    for col in df.columns:
        if df[col].dtype != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                else: df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
    return df

In [14]:
df = pd.read_csv('https://media.githubusercontent.com/media/Arannamoy/datasets/refs/heads/main/CCFD/creditcard.csv')
df = reduce_mem_usage(df)

In [15]:
df.head(5).T

,0,1,2,3,4
Time,0.000000,0.000000,1.000000,1.000000,2.000000
V1,-1.359807,1.191857,-1.358354,-0.966272,-1.158233
V2,-0.072781,0.266151,-1.340163,-0.185226,0.877737
V3,2.536347,0.166480,1.773209,1.792993,1.548718
V4,1.378155,0.448154,0.379780,-0.863291,0.403034
V5,-0.338321,0.060018,-0.503198,-0.010309,-0.407193
V6,0.462388,-0.082361,1.800499,1.247203,0.095921
V7,0.239599,-0.078803,0.791461,0.237609,0.592941
V8,0.098698,0.085102,0.247676,0.377436,-0.270533
V9,0.363787,-0.255425,-1.514654,-1.387024,0.817739


In [16]:
X = df.drop(['Class'], axis=1).values
y = df['Class'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [17]:
def train_lightgbm(X_train, y_train):
    dtrain = lgb.Dataset(X_train, label=y_train)
    params = {'objective': 'binary', 'metric': 'auc', 'device': 'gpu'} # আপনার GPU থাকলে
    model = lgb.train(params, dtrain, num_boost_round=100)
    return model

In [18]:
class FraudLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(FraudLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        out = self.fc(h_n[-1])
        return self.sigmoid(out)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [46]:
with mlflow.start_run(run_name="LightGBM_Module"):
    print("Training LightGBM...")
    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbose': -1,
        'learning_rate': 0.05,
        'device': 'gpu'
    }
    mlflow.log_params(lgb_params)
    
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_model = lgb.train(lgb_params, lgb_train, num_boost_round=100)
    lgb_preds = lgb_model.predict(X_test)
    
    mlflow.lightgbm.log_model(lgb_model, "lightgbm_model")
    mlflow.log_metric("lgb_auc", roc_auc_score(y_test, lgb_preds))

Training LightGBM...


2026/04/20 03:10:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LightGBM_Module at: http://127.0.0.1:5000/#/experiments/1/runs/c635142889bc482cbe5ddcc8b835e726
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [21]:
class FraudDataset(Dataset):
    def __init__(self, X, y, window_size=10):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.window_size = window_size

    def __len__(self):
        return len(self.X) - self.window_size + 1

    def __getitem__(self, idx):
        return self.X[idx:idx+self.window_size], self.y[idx+self.window_size-1]

In [22]:
class FraudLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super(FraudLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.sigmoid(self.fc(h_n[-1]))

In [38]:
window_size = 20
input_dim = X_train.shape[1]
model_lstm = FraudLSTM(input_dim)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_lstm.to(device)

FraudLSTM(
  (lstm): LSTM(30, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [39]:
print(f"Training LSTM on {device}")
train_loader = DataLoader(FraudDataset(X_train, y_train, window_size), batch_size=1024, shuffle=True)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)

Training LSTM on cuda


In [40]:
with mlflow.start_run(run_name="LSTM_Module"):
    mlflow.log_params({"lr": 0.001, "epochs": 50, "window_size": window_size, "dropout": 0.2})
    
    print(f"Training LSTM on {device}")
    best_loss = float('inf')
    patience = 5
    trigger_times = 0

    for epoch in range(50):
        model_lstm.train()
        epoch_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model_lstm(inputs)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        mlflow.log_metric("train_loss", avg_loss, step=epoch)
        print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")
        if avg_loss < best_loss:
            best_loss = avg_loss
            trigger_times = 0
            torch.save(model_lstm.state_dict(), 'best_lstm_model.pth')
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    mlflow.pytorch.log_model(model_lstm, "lstm_model")

Training LSTM on cuda
Epoch 1 | Loss: 0.1484
Epoch 2 | Loss: 0.0127
Epoch 3 | Loss: 0.0127
Epoch 4 | Loss: 0.0127
Epoch 5 | Loss: 0.0128
Epoch 6 | Loss: 0.0127
Epoch 7 | Loss: 0.0127


2026/04/20 03:08:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 03:08:42 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 8 | Loss: 0.0127
Early stopping at epoch 8
🏃 View run LSTM_Module at: http://0.0.0.0:5000/#/experiments/1/runs/2f2b1b0f3d8246e796c5bbc01e9c3d40
🧪 View experiment at: http://0.0.0.0:5000/#/experiments/1


In [41]:
test_loader = DataLoader(FraudDataset(X_test, y_test, window_size), batch_size=2048)

print("Predicting with LSTM")
model_lstm.eval()
lstm_preds = []
with torch.no_grad():
    for inputs, _ in test_loader:
        inputs = inputs.to(device)
        outputs = model_lstm(inputs)
        lstm_preds.extend(outputs.cpu().squeeze().tolist())

Predicting with LSTM


In [42]:
padding = [0] * (len(y_test) - len(lstm_preds))
lstm_preds_converted = np.array(padding + lstm_preds)

In [43]:
with mlflow.start_run(run_name="Hybrid_Final_Report"):
    final_preds = (0.3 * lgb_preds) + (0.7 * lstm_preds_converted)
    final_labels = (final_preds > 0.3).astype(int)
    
    auc_score = roc_auc_score(y_test, final_preds)
    mlflow.log_metric("hybrid_auc", auc_score)
    
    print("\n--- Final Hybrid Model Report ---")
    print(classification_report(y_test, final_labels))
    print(f"Hybrid AUC Score: {auc_score:.4f}")


--- Final Hybrid Model Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.95      0.56      0.71        98

    accuracy                           1.00     56962
   macro avg       0.97      0.78      0.85     56962
weighted avg       1.00      1.00      1.00     56962

Hybrid AUC Score: 0.9429
🏃 View run Hybrid_Final_Report at: http://0.0.0.0:5000/#/experiments/1/runs/55bcd68190374bdfb9df01eabb008a22
🧪 View experiment at: http://0.0.0.0:5000/#/experiments/1


In [44]:
torch.save(model_lstm.state_dict(), 'fraud_lstm_model.pth')
lgb_model.save_model('lgb_fraud_model.txt')
print("Models saved successfully!")

Models saved successfully!


In [52]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import mlflow
import mlflow.pytorch
import mlflow.xgboost


mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Fraud_Detection_Hybrid_Model")

print("Loading Data...")
# df = pd.read_csv('https://media.githubusercontent.com/media/Arannamoy/datasets/refs/heads/main/CCFD/creditcard.csv')

for col in df.columns:
    if df[col].dtype != object:
        df[col] = df[col].astype(np.float32)

X = df.drop(['Class'], axis=1).values
y = df['Class'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)


with mlflow.start_run(run_name="XGBoost_Branch"):
    print("Training XGBoost...")
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.001,
        tree_method='hist',
        device='cuda' if torch.cuda.is_available() else 'cpu',
        scale_pos_weight=10
    )
    xgb_model.fit(X_train, y_train)
    xgb_preds = xgb_model.predict_proba(X_test)[:, 1]
    
    mlflow.xgboost.log_model(xgb_model, "xgboost_model")
    mlflow.log_metric("xgb_auc", roc_auc_score(y_test, xgb_preds))


class FraudDataset(Dataset):
    def __init__(self, X, y, window_size=10):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.window_size = window_size
    def __len__(self):
        return len(self.X) - self.window_size + 1
    def __getitem__(self, idx):
        return self.X[idx:idx+self.window_size], self.y[idx+self.window_size-1]

class FraudGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.2):
        super(FraudGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True, dropout=dropout if dropout > 0 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)
   

    def forward(self, x):
        _, h_n = self.gru(x)
        out = self.dropout(h_n[-1])
        return self.fc(out)


window_size = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_gru = FraudGRU(input_dim=X_train.shape[1]).to(device)
train_loader = DataLoader(FraudDataset(X_train, y_train, window_size), batch_size=2048, shuffle=True)


pos_weight = torch.tensor([15.0]).to(device) 
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_gru.parameters(), lr=0.001)

with mlflow.start_run(run_name="GRU_Branch"):
    print(f"Training GRU on {device}...")
    model_gru.train()
    for epoch in range(50):
        epoch_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model_gru(inputs)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        mlflow.log_metric("train_loss", epoch_loss/len(train_loader), step=epoch)
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")

    mlflow.pytorch.log_model(model_gru, "gru_model")


model_gru.eval()
test_loader = DataLoader(FraudDataset(X_test, y_test, window_size), batch_size=2048, shuffle=False)
gru_preds = []

with torch.no_grad():
    for inputs, _ in test_loader:
        inputs = inputs.to(device)
    
        outputs = torch.sigmoid(model_gru(inputs))
        gru_preds.extend(outputs.cpu().squeeze().tolist())

padding = [0] * (len(y_test) - len(gru_preds))
gru_preds_final = np.array(padding + gru_preds)


with mlflow.start_run(run_name="Hybrid_XGB_GRU_Final"):
  
    final_preds = (0.4 * xgb_preds) + (0.6 * gru_preds_final)
    

    threshold = 0.3
    final_labels = (final_preds > threshold).astype(int)
    
    auc_score = roc_auc_score(y_test, final_preds)
    mlflow.log_metric("hybrid_auc", auc_score)
    mlflow.log_param("threshold", threshold)
    
    print("\n--- Final Hybrid Model Report (XGB + GRU) ---")
    print(classification_report(y_test, final_labels))
    print(f"Hybrid AUC Score: {auc_score:.4f}")

Loading Data...
Training XGBoost...


2026/04/20 03:23:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost_Branch at: http://127.0.0.1:5000/#/experiments/1/runs/65d3b644a358458089aed54b1ca9b0aa
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Training GRU on cuda...


/home/workstation-p/anaconda3/envs/conda-env-mlops-3-12/lib/python3.12/site-packages/torch/nn/modules/rnn.py:1364: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("GRU", *args, **kwargs)


Epoch 5 | Loss: 0.0213
Epoch 10 | Loss: 0.0156
Epoch 15 | Loss: 0.0112
Epoch 20 | Loss: 0.0068
Epoch 25 | Loss: 0.0035
Epoch 30 | Loss: 0.0020
Epoch 35 | Loss: 0.0012
Epoch 40 | Loss: 0.0009
Epoch 45 | Loss: 0.0007


2026/04/20 03:24:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 03:24:05 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 50 | Loss: 0.0006
🏃 View run GRU_Branch at: http://127.0.0.1:5000/#/experiments/1/runs/e1f511be25364f30a198cdfb01b9e64d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

--- Final Hybrid Model Report (XGB + GRU) ---
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     56864
         1.0       0.77      0.83      0.80        98

    accuracy                           1.00     56962
   macro avg       0.89      0.91      0.90     56962
weighted avg       1.00      1.00      1.00     56962

Hybrid AUC Score: 0.9717
🏃 View run Hybrid_XGB_GRU_Final at: http://127.0.0.1:5000/#/experiments/1/runs/54bf277f5ef748e7ad573accdb43659f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [56]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import mlflow
import mlflow.pytorch
import mlflow.xgboost
import mlflow.lightgbm
import gc

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("XGBoost_LightBGM_LSTM_GRU_Ensemble")


print("Loading and Scaling Data...")


X = df.drop(['Class'], axis=1).values.astype(np.float32)
y = df['Class'].values.astype(np.float32)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.4, random_state=42, stratify=y)
X_meta, X_test, y_meta, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)


with mlflow.start_run(run_name="XGBoost_Level0"):
    xgb_m = xgb.XGBClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.03,
        tree_method='hist', device='cuda', scale_pos_weight=12
    )
    xgb_m.fit(X_train, y_train)
    mlflow.xgboost.log_model(xgb_m, "xgb_model")


with mlflow.start_run(run_name="LightGBM_Level0"):
    lgb_m = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.03, device='gpu', 
        gpu_platform_id=0, gpu_device_id=0, verbose=-1
    )
    lgb_m.fit(X_train, y_train)
    mlflow.lightgbm.log_model(lgb_m, "lgbm_model")



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SequenceDataset(Dataset):
    def __init__(self, X, y, window_size=12):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.window_size = window_size
    def __len__(self): return len(self.X) - self.window_size + 1
    def __getitem__(self, idx):
        return self.X[idx:idx+self.window_size], self.y[idx+self.window_size-1]


class FraudLSTM(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, 64, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return torch.sigmoid(self.fc(h_n[-1]))


class FraudGRU(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, 64, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        _, h_n = self.gru(x)
        return torch.sigmoid(self.fc(h_n[-1]))

def train_neural_branch(model, name, loader):
    with mlflow.start_run(run_name=f"{name}_Level0"):
        model.to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.BCELoss()
        for epoch in range(15):
            for i, (inputs, labels) in enumerate(loader):
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                loss = criterion(model(inputs).squeeze(), labels)
                loss.backward()
                optimizer.step()
        mlflow.pytorch.log_model(model, f"{name.lower()}_model")
    return model

train_loader = DataLoader(SequenceDataset(X_train, y_train), batch_size=2048, shuffle=True)
lstm_m = train_neural_branch(FraudLSTM(X_train.shape[1]), "LSTM", train_loader)
gru_m = train_neural_branch(FraudGRU(X_train.shape[1]), "GRU", train_loader)



def get_preds(model, data):
    model.eval()
    ds = SequenceDataset(data, np.zeros(len(data)))
    loader = DataLoader(ds, batch_size=2048, shuffle=False)
    preds = []
    with torch.no_grad():
        for inputs, _ in loader:
            preds.extend(model(inputs.to(device)).cpu().squeeze().tolist())
    return np.array([0]*(len(data)-len(preds)) + preds)

print("Building Meta-Dataset...")

m_xgb = xgb_m.predict_proba(X_meta)[:, 1]
m_lgbm = lgb_m.predict_proba(X_meta)[:, 1]
m_lstm = get_preds(lstm_m, X_meta)
m_gru = get_preds(gru_m, X_meta)

X_meta_final = np.column_stack((m_xgb, m_lgbm, m_lstm, m_gru))


class MetaNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

with mlflow.start_run(run_name="Meta_Learner_NN"):
    meta_model = MetaNN().to(device)
    meta_opt = torch.optim.Adam(meta_model.parameters(), lr=0.01)
    meta_crit = nn.BCELoss()
    
    meta_x_tensor = torch.tensor(X_meta_final, dtype=torch.float32).to(device)
    meta_y_tensor = torch.tensor(y_meta, dtype=torch.float32).to(device)
    
    for _ in range(100): 
        meta_opt.zero_grad()
        loss = meta_crit(meta_model(meta_x_tensor).squeeze(), meta_y_tensor)
        loss.backward()
        meta_opt.step()
    mlflow.pytorch.log_model(meta_model, "meta_nn_model")


print("Final Evaluation on Test Set...")
t_xgb = xgb_m.predict_proba(X_test)[:, 1]
t_lgbm = lgb_m.predict_proba(X_test)[:, 1]
t_lstm = get_preds(lstm_m, X_test)
t_gru = get_preds(gru_m, X_test)

X_test_final = torch.tensor(np.column_stack((t_xgb, t_lgbm, t_lstm, t_gru)), dtype=torch.float32).to(device)
meta_model.eval()
with torch.no_grad():
    final_probs = meta_model(X_test_final).cpu().squeeze().numpy()

threshold = 0.3
final_preds = (final_probs > threshold).astype(int)

print("\n--- Ensemble Report ---")
print(classification_report(y_test, final_preds))
print(f"Final AUC Score: {roc_auc_score(y_test, final_probs):.4f}")
with mlflow.start_run(run_name="Hybrid_XGB_GRU_Final"):
    mlflow.log_metric("hybrid_auc", auc_score)
    mlflow.log_param("threshold", threshold)
gc.collect()
torch.cuda.empty_cache()

2026/04/20 03:35:04 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_LightBGM_LSTM_GRU_Ensemble' does not exist. Creating a new experiment.


Loading and Scaling Data...


2026/04/20 03:35:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost_Level0 at: http://127.0.0.1:5000/#/experiments/3/runs/9755e95624774c5ba0370e00ab23dab3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/04/20 03:35:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 03:35:09 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run GRU_Level0 at: http://127.0.0.1:5000/#/experiments/3/runs/43a0abdf39ab4309bb8aa041e6f5239c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Building Meta-Dataset...


/home/workstation-p/anaconda3/envs/conda-env-mlops-3-12/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/04/20 03:35:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 03:35:31 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


🏃 View run Meta_Learner_NN at: http://127.0.0.1:5000/#/experiments/3/runs/694e93b650ff4dc4bc8885548732e8ed
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Final Evaluation on Test Set...


/home/workstation-p/anaconda3/envs/conda-env-mlops-3-12/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Ensemble Report ---
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     56863
         1.0       0.95      0.82      0.88        99

    accuracy                           1.00     56962
   macro avg       0.98      0.91      0.94     56962
weighted avg       1.00      1.00      1.00     56962

Final AUC Score: 0.9723
🏃 View run Hybrid_XGB_GRU_Final at: http://127.0.0.1:5000/#/experiments/3/runs/c5fbf2f8b20a4676aa5aecacc2f1c9b3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
